# Kontrola kvality výstupu pipeline (QC) — GeoParquet

Tento notebook analyzuje **hlavní výstup pipeline** (`navrh/output/*_history.parquet`)
a kontroluje, jestli pipeline běžela korektně.

1. základní přehled (rozměry, časový rozsah, typy sloupců)
2. **chybějící hodnoty** — podíl NaN po sloupcích
3. **konzistence časových sloupců** (`valid_from < valid_to`, `snapshot_date == valid_from`, ...)
4. **návaznost verzí** téhož objektu (mezery/překryvy, víc než jedna otevřená verze)
5. **rozumnost hodnot** (počty podlaží, bytů, zastavěná plocha; kódy mimo číselník)
6. **geometrie** — polygon / bez polygonu, definiční bod, validita, rozsah souřadnic, mapa
7. souhrn podle obce

In [ ]:
# --- import knihoven ---

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# --- konfigurace ---

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "analyza" else Path.cwd()

# soubor k analýze:

INPUT_PARQUET = REPO_ROOT / "navrh" / "output" / "brno_area_history.parquet"

# adresář s číselníky RÚIAN
CODELIST_DIR = REPO_ROOT / "analyza" / "ciselniky"
CODELISTS = {
    "building_type_code": "CS_TYP_STAVEBNIHO_OBJEKTU.csv",
    "usage_type_code": "CE_ZPUSOB_VYUZITI_OBJEKTU.csv",
    "construction_type_code": "CE_DRUH_KONSTRUKCE.csv",
    "sewage_connection_code": "CE_PRIPOJENI_KANAL.csv",
    "gas_connection_code": "CE_PRIPOJENI_PLYNU.csv",
    "water_connection_code": "CE_PRIPOJENI_VODY.csv",
    "elevator_code": "CE_VYBAVENI_VYTAHEM.csv",
    "heating_type_code": "CE_ZPUSOB_VYTAPENI.csv",
}

DATE_COLS = ["record_valid_from", "completion_date", "snapshot_date", "valid_from", "valid_to"]
NUMERIC_COLS = ["unit_count", "floor_count", "built_up_area"]

print(f"Vstupní soubor: {INPUT_PARQUET}  (existuje: {INPUT_PARQUET.exists()})")

In [ ]:
# --- načtení ---

df = gpd.read_parquet(INPUT_PARQUET)
RAW_DTYPES = df.dtypes.astype(str)   # typy tak, jak jsou v souboru (před úpravou níž)

# datumy jsou v souboru typu date32 (Arrow); pro porovnávání níž na datetime64
for col in DATE_COLS:
    df[col] = pd.to_datetime(df[col])

IS_MULTI_OBEC = "municipality_code" in df.columns
print(f"Načteno {len(df):,} řádků, {df.shape[1]} sloupců.".replace(",", " "))
print(f"Víc obcí v souboru: {IS_MULTI_OBEC}")
df.head(3)

## 1. Základní přehled

In [ ]:
print(f"Unikátních objektů (code): {df['code'].nunique():,}".replace(",", " "))
print(f"Řádků historie celkem:     {len(df):,}".replace(",", " "))
print(f"Verzí na objekt (průměr):  {len(df) / df['code'].nunique():.2f}")
print()
print("Rozsah snapshot_date:", df['snapshot_date'].min(), "→", df['snapshot_date'].max())
print("Počet distinct snapshotů:", df['snapshot_date'].nunique())
print()
print("end_reason:")
print(df['end_reason'].value_counts(dropna=False))
if IS_MULTI_OBEC:
    print()
    print(f"Počet obcí v souboru: {df['municipality_code'].nunique()}")

import pyarrow.parquet as pq
arrow_types = {f.name: str(f.type) for f in pq.read_schema(INPUT_PARQUET)}
pd.DataFrame({"typ v souboru": pd.Series(arrow_types), "typ v pandas": RAW_DTYPES})

## 2. Chybějící hodnoty

Dvě verze: po řádcích (verzích) a po objektech (`code`) — u druhé se sloupec
počítá jako chybějící jen když chybí úplně ve všech verzích objektu

### 2a. Na úrovni řádků (verzí)

In [ ]:
missing = (df.isna().mean() * 100).sort_values(ascending=False)
missing_tbl = missing.to_frame("% chybí").round(2)
missing_tbl["počet chybí"] = (missing / 100 * len(df)).round().astype(int)
missing_tbl

### 2b. Na úrovni objektů (`code`)

In [ ]:
notna_any = df.notna().groupby(df["code"]).any()
missing_obj = (100 * (~notna_any).mean()).sort_values(ascending=False)
missing_obj_tbl = missing_obj.to_frame("% chybí").round(2)
missing_obj_tbl["počet chybí"] = (missing_obj / 100 * df["code"].nunique()).round().astype(int)
missing_obj_tbl

## 3. Konzistence časových sloupců


- `valid_from` nikdy nechybí
- `valid_to IS NULL` právě když `end_reason IS NULL` (objekt je aktuálně otevřený)
- pokud je `valid_to` vyplněné, musí platit `valid_from < valid_to`
- `snapshot_date` verze se musí rovnat jejímu `valid_from` (verze se otevírá přesně
  v okamžiku snapshotu, který ji poprvé zachytil)
- `record_valid_from` (platnost podle katastru) by neměl být pozdější než `valid_from`
  (snapshot, ve kterém jsme verzi poprvé zachytili) 
- `completion_date` (datum dokončení stavby) by mělo být rozumné datum (ne v budoucnosti)

In [ ]:
n = len(df)

def report(name, mask, extra_cols=None):
    cnt = int(mask.sum())
    pct = 100 * cnt / n if n else 0
    status = "OK" if cnt == 0 else "POZOR"
    print(f"[{status:5s}] {name}: {cnt:,} řádků ({pct:.3f} %)".replace(",", " "))
    return cnt


report(
    "valid_from chybí", df["valid_from"].isna())

report(
    "valid_to je NULL, ale end_reason ne (nebo naopak)",
    df["valid_to"].isna() != df["end_reason"].isna())

report(
    "valid_from >= valid_to (u uzavřených verzí)",
    df["valid_to"].notna() & (df["valid_from"] >= df["valid_to"]))

report(
    "snapshot_date != valid_from",
    df["snapshot_date"] != df["valid_from"])

report(
    "record_valid_from je pozdější než valid_from (verze 'z budoucnosti')",
    df["record_valid_from"] > df["valid_from"])

current_year = pd.Timestamp.now().year
bad_completion = df["completion_date"].notna() & (
    (df["completion_date"].dt.year < 1000) | (df["completion_date"].dt.year > current_year)
)
report(
    f"completion_date mimo rozsah [1000, {current_year}]", bad_completion)

end_reason_ok = {"change", "removed"}
bad_end_reason = df["end_reason"].notna() & ~df["end_reason"].isin(end_reason_ok)
report(
    f"end_reason mimo {sorted(end_reason_ok)}", bad_end_reason)

## 4. Návaznost verzí (mezery a překryvy mezi verzemi objektu)

Verze téhož `code` by na sebe měly plynule navazovat —
`valid_to` jedné verze by se mělo rovnat `valid_from` následující verze.
Kontrolujeme:

- **mezery** (`valid_to[i] < valid_from[i+1]`) — něco mezi verzemi chybí
  (typicky výpadek/chybějící měsíční snapshot)
- **překryvy** (`valid_to[i] > valid_from[i+1]`) — verze se časově kryjí
- **duplicitní verze** — stejný `code` a stejný `valid_from` ve dvou řádcích
- **víc než jedna otevřená verze** téhož `code` (víc řádků s `valid_to IS NULL`)

In [ ]:
g = df.sort_values(["code", "valid_from"]).groupby("code", sort=False)

# posun o jeden řádek v rámci skupiny -> "další verze"
df_sorted = df.sort_values(["code", "valid_from"]).reset_index(drop=True)
next_valid_from = df_sorted.groupby("code")["valid_from"].shift(-1)
same_code_next = df_sorted["code"] == df_sorted["code"].shift(-1)
this_valid_to = df_sorted["valid_to"]

has_gap = same_code_next & this_valid_to.notna() & next_valid_from.notna() & (this_valid_to < next_valid_from)
has_overlap = same_code_next & this_valid_to.notna() & next_valid_from.notna() & (this_valid_to > next_valid_from)

gap_df = df_sorted.loc[has_gap, ["code", "valid_to"]].copy()
gap_df["next_valid_from"] = next_valid_from[has_gap]
gap_df["gap_days"] = (gap_df["next_valid_from"] - gap_df["valid_to"]).dt.days

overlap_df = df_sorted.loc[has_overlap, ["code", "valid_to"]].copy()
overlap_df["next_valid_from"] = next_valid_from[has_overlap]
overlap_df["overlap_days"] = (overlap_df["valid_to"] - overlap_df["next_valid_from"]).dt.days

print(f"Mezery mezi verzemi:  {len(gap_df):,}")
print(f"Překryvy mezi verzemi: {len(overlap_df):,}")


if len(gap_df):
    print("\nRozdělení délky mezer (dny):")
    print(gap_df["gap_days"].describe().round(1))
    print("\nNejdelší mezery:")
    display(gap_df.sort_values("gap_days", ascending=False).head(10))

In [ ]:
# duplicitní verze (stejný code + stejný valid_from)
dup_mask = df.duplicated(subset=["code", "valid_from"], keep=False)
dup_df = df.loc[dup_mask, ["code", "valid_from", "valid_to", "end_reason"]].sort_values(["code", "valid_from"])
report("duplicitní dvojice (code, valid_from)", dup_mask)
if dup_mask.any():
    display(dup_df.head(20))

In [ ]:
# víc než jedna otevřená verze na objekt
open_counts = df[df["valid_to"].isna()].groupby("code").size()
multi_open = open_counts[open_counts > 1]
print(f"Objektů s víc než jednou otevřenou verzí: {len(multi_open):,}".replace(",", " "))
if len(multi_open):
    display(multi_open.sort_values(ascending=False).head(10).to_frame("počet otevřených verzí"))

In [ ]:
# kolik objektů již zaniklo a kontrola, zda některé zanikly víckrát 
episodes = df[df["end_reason"] == "removed"].groupby("code").size()
print(f"Objektů, které aspoň jednou zanikly (end_reason='removed'): {episodes.shape[0]:,}".replace(",", " "))
print(f"Z toho s víc než jedním zánikem (>1x removed): {(episodes > 1).sum():,}".replace(",", " "))

## 5. Rozmezí hodnot 

### 5a. Číselné atributy

`unit_count` (počet bytů), `floor_count` (počet podlaží) a `built_up_area`
(zastavěná plocha v m²) by měly být nezáporné a v rozumném rozsahu

In [ ]:
desc = df[NUMERIC_COLS].describe(percentiles=[.01, .5, .9, .99, .999]).T
desc

In [ ]:
SANITY_RULES = {
    # min povolené, max "rozumné" hodnoty - jen orientační horní mez)
    "unit_count": (0, 1000),
    "floor_count": (0, 40),       
    "built_up_area": (0, 100_000),  
}

for col, (lo, hi) in SANITY_RULES.items():
    below = df[col].notna() & (df[col] < lo)
    above = df[col].notna() & (df[col] > hi)
    report(f"{col} < {lo}", below)
    report(f"{col} > {hi} (orientační horní mez)", above)
    if above.any():
        display(df.loc[above, ["code", col, "snapshot_date"]].sort_values(col, ascending=False).head(5))

### 5b. Kódované atributy proti číselníkům RÚIAN

Všech osm kódovaných atributů má oficiální
číselník RÚIAN stažený v `analyza/ciselniky/` (staženo ze
`services.cuzk.gov.cz/sestavy/cis/`) 


In [ ]:
def load_codelist(fname: str) -> set:
    path = CODELIST_DIR / fname
    if not path.exists():
        print(f"  (číselník {fname} nenalezen, sekce se přeskakuje)")
        return None
    cl = pd.read_csv(path, sep=";", encoding="cp1250")
    return set(cl["KOD"].astype(int))


for col, fname in CODELISTS.items():
    if col not in df.columns:
        continue
    valid_codes = load_codelist(fname)
    if valid_codes is None:
        continue
    present = df[col].dropna().astype(int)
    bad = present[~present.isin(valid_codes)]
    pct = 100 * len(bad) / len(present) if len(present) else 0
    status = "OK" if len(bad) == 0 else "POZOR"
    print(f"[{status:5s}] {col}: {len(bad):,} hodnot mimo číselník ({pct:.3f} %)".replace(",", " "))
    if len(bad):
        display(bad.value_counts().head(10).to_frame("počet"))

## 6. Geometrie

`geometry` je půdorys objektu (`OriginalniHranice`), `null` když ho objekt ve
zdroji nemá. `reference_point` je definiční bod. Neplatné polygony pipeline
**neopravuje ani nezahazuje** — řeší je až čištění, proto je tady jen počítáme.

In [ ]:
has_polygon = df.geometry.notna() & ~df.geometry.is_empty
has_point = df["reference_point"].notna()

print(f"Řádků s polygonem:             {has_polygon.mean() * 100:.1f} %")
print(f"Řádků jen s definičním bodem:  {(~has_polygon & has_point).mean() * 100:.2f} %")
print(f"Řádků bez jakékoliv geometrie: {(~has_polygon & ~has_point).mean() * 100:.2f} %")
print(f"Řádků s polygonem, ale bez bodu: {(has_polygon & ~has_point).sum():,}".replace(",", " "))

obj_has_polygon = has_polygon.groupby(df["code"]).any()
print(f"Objektů s polygonem aspoň v jedné verzi: {obj_has_polygon.mean() * 100:.1f} % "
      f"({obj_has_polygon.sum():,} z {len(obj_has_polygon):,})".replace(",", " "))

In [ ]:
from shapely.validation import explain_validity

poly = df.loc[has_polygon]
invalid = poly[~poly.geometry.is_valid]
report("neplatný polygon (is_valid == False)", ~poly.geometry.is_valid)
if len(invalid):
    reasons = invalid.geometry.map(explain_validity)
    display(invalid.assign(duvod=reasons)[["code", "valid_from", "duvod"]].head(10))
    print(reasons.str.split("[").str[0].value_counts().to_string())

report("polygon s nulovou plochou", poly.geometry.area == 0)
print(f"[info ] polygonů z více částí (MultiPolygon): {int((poly.geometry.geom_type == 'MultiPolygon').sum())}")

# bounding box ČR v S-JTSK (Krovak East North) - orientační
CZ_X, CZ_Y = (-950_000, -400_000), (-1_250_000, -900_000)
for name, geoms in [("geometry", poly.geometry), ("reference_point", df.loc[has_point, "reference_point"])]:
    b = geoms.bounds
    out = (b["minx"] < CZ_X[0]) | (b["maxx"] > CZ_X[1]) | (b["miny"] < CZ_Y[0]) | (b["maxy"] > CZ_Y[1])
    print(f"[{'OK' if not out.any() else 'POZOR':5s}] {name} mimo bbox ČR (S-JTSK): {int(out.sum())}")

In [ ]:
# definiční bod by měl ležet v (nebo těsně u) půdorysu stejné verze
both = df.loc[has_polygon & has_point]
dist = both.geometry.distance(gpd.GeoSeries(both["reference_point"], crs=df.crs))
far = dist > 5   # m
print(f"Řádků s bodem dál než 5 m od polygonu: {int(far.sum()):,} z {len(both):,}".replace(",", " "))
if far.any():
    display(both.loc[far, ["code", "valid_from"]].assign(vzdalenost_m=dist[far].round(1))
            .sort_values("vzdalenost_m", ascending=False).head(10))

In [ ]:
# mapa jedné obce: aktuálně otevřené verze, problémové řádky červeně
MAP_MUNICIPALITY = "582786"   # Brno
POINT_TOLERANCE_M = 5         # stejný práh jako výše

in_obec = (df["municipality_code"] == MAP_MUNICIPALITY) if IS_MULTI_OBEC else pd.Series(True, index=df.index)
current = df[in_obec & df["valid_to"].isna()]
cur_poly = current[has_polygon.loc[current.index]]
cur_point_only = current[~has_polygon.loc[current.index] & has_point.loc[current.index]]

# problémové řádky: neplatný polygon, nulová plocha, bod dál než práh od polygonu
bad_valid = ~cur_poly.geometry.is_valid
bad_area = cur_poly.geometry.area == 0
pt = cur_poly[has_point.loc[cur_poly.index]]
bad_far = pd.Series(False, index=cur_poly.index)
bad_far.loc[pt.index] = pt.geometry.distance(gpd.GeoSeries(pt["reference_point"], crs=df.crs)) > POINT_TOLERANCE_M
problem = cur_poly[bad_valid | bad_area | bad_far]

fig, ax = plt.subplots(figsize=(11, 11))
cur_poly.plot(ax=ax, color="darkgrey", linewidth=0)
if len(problem):
    problem.plot(ax=ax, color="red", linewidth=0)
    problem.boundary.plot(ax=ax, color="red", linewidth=0.8)   # na mapě jsou malé, zvýrazni obrysem
# objekty bez polygonu, jen s definičním bodem
if len(cur_point_only):
    cur_point_only["reference_point"].plot(ax=ax, color="red", marker="x", markersize=6)

name = current["municipality_name"].iat[0] if IS_MULTI_OBEC and len(current) else MAP_MUNICIPALITY
ax.set_title(
    f"{name}: otevřené verze s polygonem ({len(cur_poly):,}), "
    f"{len(problem):,} invalid, {len(cur_point_only):,} jen bod".replace(",", " ")
)
ax.set_aspect("equal")
plt.show()

print(f"neplatný polygon: {int(bad_valid.sum())}, nulová plocha: {int(bad_area.sum())}, "
      f"bod > {POINT_TOLERANCE_M} m od polygonu: {int(bad_far.sum())}")

## 7. Souhrn podle obce

Všechny sloupce jsou **na úrovni objektu**, ne řádku (jen když daný atribut chybí
ve všech jeho verzích)

- `chybi_dokonceni_pct` — objekt nemá `completion_date` v žádné verzi
- `chybi_polygon_pct` — objekt nemá půdorys v žádné verzi
- `chybi_geometrie_pct` — objekt nemá v žádné verzi ani polygon, ani definiční bod

In [ ]:
obec_cols = ["municipality_code", "municipality_name"] if IS_MULTI_OBEC else []

OBEC_NAZEV_LOOKUP = {"539309": "Chýně"}

dokonceni_chybi_obj = ~df["completion_date"].notna().groupby(df["code"]).any()
polygon_chybi_obj = ~has_polygon.groupby(df["code"]).any()
geometrie_chybi_obj = ~(has_polygon | has_point).groupby(df["code"]).any()

obj_tbl = df.drop_duplicates("code").set_index("code")[obec_cols]
obj_tbl = obj_tbl.assign(chybi_dokonceni=dokonceni_chybi_obj,
                         chybi_polygon=polygon_chybi_obj,
                         chybi_geometrie=geometrie_chybi_obj)

group_cols = obec_cols or [lambda _: "celkem"]
gb = obj_tbl.groupby(group_cols)

per_obec = gb.agg(
    objektu=("chybi_dokonceni", "size"),
    chybi_dokonceni_pct=("chybi_dokonceni", lambda s: round(100 * s.mean(), 1)),
    chybi_polygon_pct=("chybi_polygon", lambda s: round(100 * s.mean(), 1)),
    chybi_geometrie_pct=("chybi_geometrie", lambda s: round(100 * s.mean(), 1)),
)

radku = df.groupby(group_cols).size().rename("radku") if obec_cols else pd.Series({"celkem": len(df)}, name="radku")
per_obec = radku.to_frame().join(per_obec).reset_index()
if not obec_cols:
    per_obec = per_obec.drop(columns=["index"])
    kody = df["source_file"].str.extract(r"_OB_(\d+)_UKSH")[0].dropna()
    obec_kod = kody.mode().iat[0] if not kody.empty else "?"
    per_obec.insert(0, "municipality_code", obec_kod)
    per_obec.insert(1, "municipality_name", OBEC_NAZEV_LOOKUP.get(obec_kod, "?"))
per_obec = per_obec.sort_values("radku", ascending=False)

print(f"Počet obcí v souboru: {len(per_obec)}")
display(per_obec)